In [0]:
%pip install xgboost==3.2.0

In [0]:
%restart_python

In [0]:
# Importing libraries
import numpy as np
import pandas as pd
from scipy import stats
from pyspark.sql.functions import (
    col, count, avg, stddev, when, lit,
    current_timestamp, date_trunc, percentile_approx
)
from pyspark.sql.types import DoubleType, FloatType
from pyspark.ml.functions import vector_to_array
import mlflow
from mlflow.tracking import MlflowClient


def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()


def get_production_threshold(dataset_name):
    """Extracts best_threshold from MLflow"""
    client = MlflowClient()
    
    # Username definition
    username = spark.sql("SELECT current_user()").collect()[0][0]
        
    try:
        experiment_path = f"/Users/{username}/supervised_models"
        experiment = mlflow.get_experiment_by_name(experiment_path)
        
        if experiment is None:
            print(f"  ⚠️ Experiment not found in {experiment_path}. Using default 0.5")
            return 0.5

        runs = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string=f"tags.mlflow.runName = 'XGB_{dataset_name}'",
            order_by=["start_time DESC"],
            max_results=1
        )

        if len(runs) > 0 and "metrics.best_threshold" in runs.columns:
            
            threshold = float(runs.iloc[0]["metrics.best_threshold"])
            print(f"  🎯 Threshold loaded for {dataset_name}: {threshold:.4f}")
            return threshold
        else:
            print(f"  ⚠️ Threshold not found for {dataset_name} (using default 0.5)")
            return 0.5
            
    except Exception as e:
        print(f"  ⚠️ Error conecting with MLflow: {e}. Usando default 0.5")
        return 0.5

# Scoring and saving predictions. This simulates a production scoring table that would be updated daily

import xgboost as xgb

def score_and_save(train_table, test_table, feature_names,
                   output_table, dataset_name):
    """
    Score test set by loading the registered model from MLflow Registry.
    """
    train_df = spark.table(train_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))
    test_df  = spark.table(test_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))

    def extract(df):
        arr = df.withColumn("features_arr", vector_to_array("features"))
        pdf = arr.select("features_arr", "is_fraud").toPandas()
        return (np.array(pdf["features_arr"].tolist()),
                pdf["is_fraud"].values)

    X_train, y_train = extract(train_df)
    X_test,  y_test  = extract(test_df)

    client = MlflowClient()

    registered_name = f"workspace.ml_layer.Modelo_Fraude_XGB_{dataset_name.capitalize()}"    
    
    try:
        # Searching model versions registerd
        versions = client.search_model_versions(f"name='{registered_name}'")
        
        if not versions:
            raise RuntimeError(f"Model {registered_name} not found in Registry.")
            
        # Extracting latest version
        latest_version = max([int(v.version) for v in versions])
        
        # Building uri with model version found
        model_uri = f"models:/{registered_name}/{latest_version}"
        print(f"  ⬇️ Loading model version {latest_version} from Unity Catalog: {model_uri}")
        
        model = mlflow.xgboost.load_model(model_uri)
        
    except Exception as e:
        raise RuntimeError(f"Model {registered_name} upload failed. Error: {e}")

    # Direct inference
    production_threshold = get_production_threshold(dataset_name)

    scores      = model.predict_proba(X_test)[:, 1]
    predictions = (scores >= production_threshold).astype(int)

    # Build scored DataFrame
    scored_pdf = pd.DataFrame({
        "fraud_score"    : scores,
        "predicted_fraud": predictions,
        "actual_fraud"   : y_test,
        "correct"        : (predictions == y_test).astype(int),
        "threshold_used"  : production_threshold,
        "dataset"        : dataset_name,
        "scored_at"      : pd.Timestamp.now()
    })

    # Add risk tier
    scored_pdf["risk_tier"] = pd.cut(
        scored_pdf["fraud_score"],
        bins  = [0, 0.3, 0.6, 0.8, 1.0],
        labels= ["low", "medium", "high", "critical"]
    ).astype(str)

    # Save to Delta
    scored_sdf = spark.createDataFrame(scored_pdf)
    scored_sdf = scored_sdf \
        .withColumn("fraud_score", col("fraud_score").cast(FloatType())) \
        .withColumn("actual_fraud", col("actual_fraud").cast("double"))
    
    scored_sdf.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"workspace.ml_layer.{output_table}")

    print(f"  ✅ Scored {len(scored_pdf):,} rows → {output_table}")
    
    return model, X_train, X_test, y_train, y_test, scores


print("Scoring applications...")
(xgb_app, 
 X_app_train_raw, X_app_test_raw, 
 y_app_train, y_app_test, 
 app_scores) = score_and_save(
    "workspace.ml_layer.application_train_features",
    "workspace.ml_layer.application_test_features",
    APP_FEATURE_NAMES := [
        "log_income", "address_stability", "under_25",
        "name_email_similarity", "days_since_request",
        "zip_count_4w", "payment_type_index"
    ],
    "scored_applications", "applications"
)

print("Scoring transactions...")
(xgb_txn, 
 X_txn_train_raw, X_txn_test_raw, 
 y_txn_train, y_txn_test, 
 txn_scores) = score_and_save(
    "workspace.ml_layer.transaction_train_features",
    "workspace.ml_layer.transaction_test_features",
    TXN_FEATURE_NAMES := [
        "log_amount", "amount_balance_ratio", "transaction_hour",
        "transaction_dayofweek", "merchant_fraud_rate",
        "customer_avg_transaction_amount", "merchant_category_index",
        "device_type_index", "channel_grouped_index",
        "location_city_grouped_index"
    ],
    "scored_transactions", "transactions"
)

# Performance metrics table

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)

def compute_performance_metrics(y_true, y_proba, dataset_name,
                                 threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "dataset"          : dataset_name,
        "roc_auc"          : round(roc_auc_score(y_true, y_proba), 4),
        "pr_auc"           : round(average_precision_score(y_true, y_proba), 4),
        "precision"        : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall"           : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1"               : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "threshold"        : threshold,
        "true_positives"   : int(tp),
        "false_positives"  : int(fp),
        "true_negatives"   : int(tn),
        "false_negatives"  : int(fn),
        "fraud_rate_actual": round(float(y_true.mean()), 4),
        "fraud_rate_pred"  : round(float(y_pred.mean()), 4),
        "evaluated_at"     : pd.Timestamp.now().isoformat()
    }

print("\nUploading production threshold from MLflow...")

app_threshold = get_production_threshold("applications")
txn_threshold = get_production_threshold("transactions")

metrics_app = compute_performance_metrics(
    y_app_test, app_scores, "applications", threshold=app_threshold
)
metrics_txn = compute_performance_metrics(
    y_txn_test, txn_scores, "transactions", threshold=txn_threshold
)

metrics_df = pd.DataFrame([metrics_app, metrics_txn])
display(metrics_df)

spark.createDataFrame(metrics_df).write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.model_performance_metrics")

print("Performance metrics saved")

# Drift Detection analysis
# PSI (Population Stability Index)
# PSI < 0.1  → no drift
# PSI 0.1–0.2 → moderate drift, monitor
# PSI > 0.2  → significant drift, retrain

def compute_psi(expected, actual, buckets=10):
    """
    Population Stability Index between training and test distributions.
    expected = training feature values
    actual   = test/production feature values
    """
    # Create bins from expected distribution
    breakpoints = np.nanpercentile(expected,
                                   np.linspace(0, 100, buckets + 1))
    breakpoints  = np.unique(breakpoints)

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts   = np.histogram(actual,   bins=breakpoints)[0]

    # Avoid zero counts
    expected_pct = np.where(expected_counts == 0, 0.0001,
                            expected_counts / len(expected))
    actual_pct   = np.where(actual_counts == 0, 0.0001,
                            actual_counts / len(actual))

    psi = np.sum((actual_pct - expected_pct) *
                 np.log(actual_pct / expected_pct))
    return float(psi)


def run_drift_detection(X_train, feature_names, dataset_name):
    """Compute PSI for all features and classify drift severity."""
    drift_records = []
    
    # Match feature names to actual array shape
    actual_num_features = X_train.shape[1]
    feature_names_adjusted = feature_names[:actual_num_features]

    # Split training set into historical baseline vs recent data
    
    split_idx     = int(len(X_train) * 0.7)
    X_baseline    = X_train[:split_idx]   # historical → expected distribution
    X_recent      = X_train[split_idx:]   # recent → actual distribution to compare

    print(f"\n  Drift baseline size : {len(X_baseline):,} rows (older 70%)")
    print(f"  Drift comparison    : {len(X_recent):,} rows (recent 30%)")

    for i, feat in enumerate(feature_names_adjusted):
        psi = compute_psi(X_baseline[:, i], X_recent[:, i])

        if psi < 0.1:
            status = "✅ Stable"
            action = "None"
        elif psi < 0.2:
            status = "⚠️  Moderate drift"
            action = "Monitor closely"
        else:
            status = "🔴 Significant drift"
            action = "Consider retraining"

        drift_records.append({
            "feature"   : feat,
            "psi"       : round(psi, 4),
            "status"    : status,
            "action"    : action,
            "dataset"   : dataset_name,
            "checked_at": pd.Timestamp.now().isoformat()
        })

    drift_df = pd.DataFrame(drift_records).sort_values(
        "psi", ascending=False
    ).reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"  DRIFT DETECTION — {dataset_name}")
    print(f"{'='*60}")
    display(drift_df)

    return drift_df


drift_app = run_drift_detection(
    X_app_train_raw,
    APP_FEATURE_NAMES, "applications"
)
drift_txn = run_drift_detection(
    X_txn_train_raw,
    TXN_FEATURE_NAMES, "transactions"
)

drift_all = pd.concat([drift_app, drift_txn], ignore_index=True)
spark.createDataFrame(drift_all).write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.drift_detection_results")

print("Drift detection results saved")

# Baseline metrics management

from mlflow.tracking import MlflowClient

def get_or_create_baseline(dataset_name, model_name="XGBoost"):
    """
    Get baseline metrics for a model. Auto-captures from MLflow on first run,
    persists to Delta for subsequent comparisons.
    
    Returns: dict with baseline_pr_auc, baseline_recall, baseline_roc_auc, baseline_f1
    """
    baseline_table = "workspace.ml_layer.model_baseline_metrics"
    
    # Checking if baseline exists
    try:
        existing = spark.table(baseline_table) \
            .filter(f"dataset = '{dataset_name}' AND model = '{model_name}'") \
            .toPandas()
        
        if len(existing) > 0:
            baseline = existing.iloc[0].to_dict()
            print(f"  ✅ Using existing baseline for {dataset_name} "
                  f"(captured {baseline.get('captured_at', 'unknown')})")
            return baseline
    except Exception:
        pass

    # Auto-capture baseline from MLflow
    print(f"  📌 FIRST-TIME baseline capture for {dataset_name}")
    print(f"     This baseline will be locked until refresh_baseline() is called")
    print(f"     Future monitoring runs will compare against this snapshot")
    
    client     = MlflowClient()
    username   = spark.sql("SELECT current_user()").collect()[0][0]
    experiment = mlflow.get_experiment_by_name(f"/Users/{username}/supervised_models")
    
    if experiment is None:
        raise RuntimeError("Cannot capture baseline: supervised_models experiment not found")

    # Find most recent successful XGBoost run for this dataset
    run_name = f"XGB_{dataset_name}"
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
        order_by=["start_time DESC"],
        max_results=1
    )
    
    if len(runs) == 0:
        raise RuntimeError(f"No XGBoost runs found for {dataset_name}")
    
    latest = runs.iloc[0]
    
    baseline = {
        "model"            : model_name,
        "dataset"          : dataset_name,
        "baseline_pr_auc"  : float(latest.get("metrics.pr_auc",  0)),
        "baseline_roc_auc" : float(latest.get("metrics.roc_auc", 0)),
        "baseline_recall"  : float(latest.get("metrics.recall",  0)),
        "baseline_f1"      : float(latest.get("metrics.f1",      0)),
        "baseline_run_id"  : latest.get("run_id"),
        "captured_at"      : pd.Timestamp.now().isoformat(),
        "captured_from"    : "auto_mlflow"
    }
    
    # Persist for future runs
    baseline_df = pd.DataFrame([baseline])
    
    try:
        existing = spark.table(baseline_table).toPandas()
        combined = pd.concat([existing, baseline_df], ignore_index=True)
    except Exception:
        combined = baseline_df
    
    spark.createDataFrame(combined).write.format("delta") \
        .mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(baseline_table)
    
    print(f"  ✅ Captured baseline from run {baseline['baseline_run_id'][:8]}")
    print(f"     PR-AUC: {baseline['baseline_pr_auc']:.4f} | "
          f"Recall: {baseline['baseline_recall']:.4f}")
    
    return baseline

# Retraining requirements assessment

def assess_retraining_requirements(metrics_df, drift_df, dataset_name):
    """
    Combines performance degradation + drift signals into
    a retraining recommendation.
    """
    m = metrics_df[metrics_df["dataset"] == dataset_name].iloc[0]
    baseline = get_or_create_baseline(dataset_name, "XGBoost")

    # Thresholds 
    DEGRADATION_TOLERANCE = 0.15
    PSI_ALERT             = 0.20
    
    pr_auc_threshold = baseline["baseline_pr_auc"] * (1 - DEGRADATION_TOLERANCE)
    recall_threshold = baseline["baseline_recall"] * (1 - DEGRADATION_TOLERANCE)

    #Drift check  

    dataset_drift = drift_df[drift_df["dataset"] == dataset_name]
    drifted_features = dataset_drift[
        dataset_drift["psi"] > PSI_ALERT
    ]["feature"].tolist()

    triggers = []

    if m["pr_auc"] < pr_auc_threshold:
        degradation_pct = (1 - m["pr_auc"] / baseline["baseline_pr_auc"]) * 100
        triggers.append(
            f"PR-AUC degraded {degradation_pct:.1f}% from baseline "
            f"({baseline['baseline_pr_auc']:.4f} → {m['pr_auc']:.4f})"
        )
    
    if m["recall"] < recall_threshold:
        degradation_pct = (1 - m["recall"] / baseline["baseline_recall"]) * 100
        triggers.append(
            f"Recall degraded {degradation_pct:.1f}% from baseline "
            f"({baseline['baseline_recall']:.4f} → {m['recall']:.4f})"
        )
    
    if drifted_features:
        triggers.append(
            f"Feature drift detected: {', '.join(drifted_features)}"
        )

    recommendation = "🔴 RETRAIN" if triggers else "🟢 NO ACTION NEEDED"

    result = {
        "dataset"              : dataset_name,
        "recommendation"       : recommendation,
        "current_pr_auc"       : round(m["pr_auc"], 4),
        "baseline_pr_auc"      : round(baseline["baseline_pr_auc"], 4),
        "pr_auc_threshold"     : round(pr_auc_threshold, 4),
        "current_recall"       : round(m["recall"], 4),
        "baseline_recall"      : round(baseline["baseline_recall"], 4),
        "recall_threshold"     : round(recall_threshold, 4),
        "degradation_tolerance": f"{int(DEGRADATION_TOLERANCE*100)}%",
        "drifted_features"     : ", ".join(drifted_features) if drifted_features else "None",
        "triggers"             : " | ".join(triggers) if triggers else "None",
        "baseline_run_id"      : baseline.get("baseline_run_id", "")[:8],
        "assessed_at"          : pd.Timestamp.now().isoformat()
    }

    print(f"\n{'='*60}")
    print(f"  RETRAINING ASSESSMENT — {dataset_name}")
    print(f"{'='*60}")
    print(f"  Recommendation : {recommendation}")
    print(f"  Tolerance      : {DEGRADATION_TOLERANCE*100:.0f}% from baseline")
    print(f"  Baseline PR-AUC: {baseline['baseline_pr_auc']:.4f}")
    print(f"  Current PR-AUC : {m['pr_auc']:.4f}")
    print(f"  Threshold      : {pr_auc_threshold:.4f}")
    if triggers:
        for t in triggers:
            print(f"    → {t}")
    
    return result


retraining_app = assess_retraining_requirements(
    metrics_df, drift_all, "applications"
)
retraining_txn = assess_retraining_requirements(
    metrics_df, drift_all, "transactions"
)

retraining_df = pd.DataFrame([retraining_app, retraining_txn])
spark.createDataFrame(retraining_df).write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.retraining_requirements")

print("\nRetraining requirements saved")


# Baseline refresher when optimization takes place

def refresh_baseline(dataset_name, model_name="XGBoost"):
    """
    Force-refresh baseline from most recent MLflow run.
    For using AFTER intentional retraining.
    
    Usage:
        refresh_baseline("transactions")
        refresh_baseline("applications")
    """
    baseline_table = "workspace.ml_layer.model_baseline_metrics"
    
    # Delete existing baseline
    try:
        existing = spark.table(baseline_table).toPandas()
        cleaned  = existing[
            ~((existing["dataset"] == dataset_name) & 
              (existing["model"] == model_name))
        ]
        spark.createDataFrame(cleaned).write.format("delta") \
            .mode("overwrite").option("overwriteSchema", "true") \
            .saveAsTable(baseline_table)
    except Exception:
        pass
    
    # Re-capture from MLflow
    new_baseline = get_or_create_baseline(dataset_name, model_name)
    print(f"\n✅ Baseline refreshed for {dataset_name}")
    return new_baseline


# Manual usage example:
# refresh_baseline("applications")
# refresh_baseline("transactions")

# mlflow logging results

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/monitoring")

with mlflow.start_run(run_name=f"monitoring_{pd.Timestamp.now().strftime('%Y%m%d_%H%M')}"):
    # Log current performance
    for metric in metrics_app, metrics_txn:
        prefix = metric["dataset"]
        mlflow.log_metric(f"{prefix}_pr_auc",    metric["pr_auc"])
        mlflow.log_metric(f"{prefix}_recall",    metric["recall"])
        mlflow.log_metric(f"{prefix}_precision", metric["precision"])
    
    # Log drift summary
    for dataset in ["applications", "transactions"]:
        max_psi = drift_all[drift_all["dataset"] == dataset]["psi"].max()
        mlflow.log_metric(f"{dataset}_max_psi", float(max_psi))
    
    # Log retraining decisions as tags (searchable)
    mlflow.set_tag("retrain_applications", retraining_app["recommendation"])
    mlflow.set_tag("retrain_transactions", retraining_txn["recommendation"])